# Weapon check

**Question:** does this crowd view contain a shape like a firearm or a blade?

The answer the desk is allowed to show is **Possible firearm** or **Possible blade**. It is not a statement that a person is armed. The officer watches the clip and then chooses Deploy police.

On the sample counts, which are all 40 or more, this model returns `blade` and `strength: review`. A count under 40 returns `none` and does not, by itself, open the crowd incident.


In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from cityvigilance.runtime import explain, where_this_runs
from cityvigilance.sample import check_event, run_model


In [2]:
print(explain())


Machine: cursor
System: Linux-6.12.94+-x86_64-with-glibc2.39
Python: 3.12.3
NVIDIA tool: not installed
PyTorch installed: False
CUDA available: False
Vision API connected: False
Model source: sample
This notebook runs on: this process, CPU only

The command-centre page is a separate Node process. It serves the desk
and plays the same sample findings in the browser. It does not send
frames to a GPU or to a vision API.

Your own computer's GPU is not used when you watch the demo from Cursor.
If you clone the repo and run it locally, the sample models still use
the CPU only, until you replace one of them with a real model.


In [3]:
print(f"{'tick':<6}{'kind':<10}{'strength':<10}source")
for tick in range(8):
    event = run_model("weapon", tick)
    check_event(event)
    print(f"{tick:<6}{event['finding']['kind']:<10}{event['strength']:<10}{event['source']}")


tick  kind      strength  source
0     blade     review    sample
1     blade     review    sample
2     blade     review    sample
3     blade     review    sample
4     blade     review    sample
5     blade     review    sample
6     blade     review    sample
7     blade     review    sample


## Below the sample threshold

A quiet scene must not raise a weapon. This cell checks that rule with a hand-built event, which is the shape a real model should return when it sees nothing.


In [4]:
quiet = {
    "id": "cam-crowd:weapon:quiet",
    "cameraId": "cam-crowd",
    "model": "weapon",
    "at": "2026-11-02T00:00:00Z",
    "sequence": 0,
    "source": "sample",
    "strength": "review",
    "finding": {"type": "weapon", "kind": "none"},
}
check_event(quiet)
assert quiet["finding"]["kind"] in {"firearm", "blade", "none"}
print("quiet weapon event ok", quiet["finding"])


quiet weapon event ok {'type': 'weapon', 'kind': 'none'}


## Slot for a real model

A real weapon model must still return `kind` as `firearm`, `blade`, or `none`, and `strength` as `review` when a person should look. Do not return a person's identity.


In [5]:
def my_weapon(sequence: int, at: str) -> dict:
    return run_model("weapon", sequence, at)

result = my_weapon(0, "2026-11-02T00:00:00Z")
check_event(result)
assert result["finding"]["kind"] in {"firearm", "blade", "none"}
print("weapon event ok", result["finding"])


weapon event ok {'type': 'weapon', 'kind': 'blade'}
